# 04d — Method: TF-IDF + BERT Hybrid (Enhanced)
## UniMatch: Arabic Student Advising Chatbot

### Motivation
Previous hybrid (BM25 + BERT) underperformed because BM25 word-level
tokenization misses Arabic morphological patterns.

**New approach:** Replace BM25 with TF-IDF char n-grams as the first stage.
TF-IDF char n-grams are better suited for Arabic since they capture
subword patterns regardless of morphological variation.

### Two-Stage Pipeline
```
Query
  ↓
Stage 1: TF-IDF (char n-grams) → top-k candidates (fast + Arabic-aware)
  ↓
Stage 2: BERT re-ranks candidates (semantic precision)
  ↓
Final Answer
```

### Expected Improvement
| Method | Accuracy@1 |
|--------|----------|
| BM25 + BERT (old hybrid) | ~69% |
| TF-IDF alone | 90.09% |
| BERT alone | 92.59% |
| **TF-IDF + BERT (this notebook)** | **> 92.59% ?** |

## Cell 1 — Install

In [15]:
!pip install -q sentence-transformers scikit-learn pandas matplotlib

## Cell 2 — Mount Drive & Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR   = '/content/drive/MyDrive/Student_chatbot/'
DATA_DIR   = os.path.join(BASE_DIR, 'data/')
MODEL_DIR  = os.path.join(BASE_DIR, 'models/')
RESULT_DIR = os.path.join(BASE_DIR, 'results/')

os.makedirs(RESULT_DIR, exist_ok=True)

print('Checking required files:')
for folder, f in [
    (DATA_DIR,  'faq_enhanced.csv'),
    (MODEL_DIR, 'bert_embeddings_v2.npy'),
]:
    path = os.path.join(folder, f)
    print(f'  {f:30s} {"✅" if os.path.exists(path) else "❌ MISSING"}')

## Cell 3 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

print('Imports done ✅')

## Cell 4 — Load Dataset

In [ ]:
# Index = ALL questions
df_index = pd.read_csv(os.path.join(DATA_DIR, 'faq_enhanced.csv'))

# Build test queries — 4 paraphrases per question
long_rows = []
for _, row in df_index.iterrows():
    for col in ['question_alt', 'question_alt2',
                'question_alt3', 'question_alt4']:
        if col in row and pd.notna(row[col]):
            long_rows.append({
                'question': row['question'],
                'answer'  : row['answer'],
                'query'   : row[col],
            })

df_test = pd.DataFrame(long_rows)

print(f'Index  : {len(df_index)} questions ✅')
print(f'Test   : {len(df_test)} queries ✅')

## Cell 5 — Build TF-IDF Index (Stage 1)

In [ ]:
# char n-grams — best for Arabic morphology
vectorizer = TfidfVectorizer(
    analyzer     = 'char_wb',
    ngram_range  = (2, 4),
    min_df       = 1,
    sublinear_tf = True
)

tfidf_matrix = vectorizer.fit_transform(df_index['question'])

print(f'TF-IDF index built ✅')
print(f'  Vocabulary size : {len(vectorizer.vocabulary_):,}')
print(f'  Matrix shape    : {tfidf_matrix.shape}')

## Cell 6 — Load BERT Model & Embeddings (Stage 2)

In [ ]:
MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'
print(f'Loading BERT: {MODEL_NAME}')
model = SentenceTransformer(MODEL_NAME)

emb_path = os.path.join(MODEL_DIR, 'bert_embeddings_v2.npy')
if os.path.exists(emb_path):
    bert_embeddings = np.load(emb_path)
    print(f'Embeddings loaded : {bert_embeddings.shape} ✅')
else:
    print('Computing embeddings...')
    bert_embeddings = model.encode(
        df_index['question'].tolist(),
        batch_size=32, show_progress_bar=True,
        normalize_embeddings=True
    )
    np.save(emb_path, bert_embeddings)
    print(f'Embeddings saved  : {bert_embeddings.shape} ✅')

## Cell 7 — TF-IDF + BERT Hybrid Retrieval Function

In [ ]:
def retrieve_tfidf_bert(query, vectorizer, tfidf_matrix,
                        model, bert_embeddings, df_index,
                        tfidf_top_k=10, final_top_k=5):
    """
    Two-stage hybrid:
    Stage 1: TF-IDF char n-grams → top-k candidates
    Stage 2: BERT re-ranks candidates semantically

    Args:
        tfidf_top_k : candidates from TF-IDF stage
        final_top_k : final results after BERT re-ranking
    """
    # ── Stage 1: TF-IDF ──────────────────────────────────────
    q_vec        = vectorizer.transform([query])
    tfidf_scores = cosine_similarity(q_vec, tfidf_matrix)[0]
    tfidf_top_idx = np.argsort(tfidf_scores)[::-1][:tfidf_top_k]

    # ── Stage 2: BERT re-ranking ──────────────────────────────
    cand_embeddings = bert_embeddings[tfidf_top_idx]
    q_emb           = model.encode([query], normalize_embeddings=True)
    bert_scores     = cosine_similarity(q_emb, cand_embeddings)[0]

    reranked_idx  = np.argsort(bert_scores)[::-1][:final_top_k]
    final_indices = tfidf_top_idx[reranked_idx]

    top_k_results = [
        (df_index.iloc[i]['question'],
         float(bert_scores[reranked_idx[j]]))
        for j, i in enumerate(final_indices)
    ]

    best_global = final_indices[0]
    best_score  = float(bert_scores[reranked_idx[0]])

    return (
        df_index.iloc[best_global]['answer'],
        best_score,
        df_index.iloc[best_global]['question'],
        top_k_results
    )

print('TF-IDF + BERT hybrid function defined ✅')

## Cell 8 — Test on Sample Queries

In [ ]:
sample_queries = df_test['query'].sample(4, random_state=42).tolist()

print('TF-IDF + BERT Hybrid — Sample Results')
print('='*65)
for q in sample_queries:
    answer, score, matched, _ = retrieve_tfidf_bert(
        q, vectorizer, tfidf_matrix,
        model, bert_embeddings, df_index)
    print(f'\n  Query   : {q}')
    print(f'  Matched : {matched}')
    print(f'  Score   : {score:.4f}')
    print(f'  Answer  : {answer[:80]}...')
print('\n' + '='*65)

## Cell 9 — TF-IDF top-k Ablation
Find the best candidate pool size k.

In [ ]:
sample_df = df_test.sample(40, random_state=42)
k_values  = [3, 5, 10, 15, 20, 30, 50]
k_acc     = []

for k in k_values:
    correct = 0
    for _, row in sample_df.iterrows():
        answer, _, _, _ = retrieve_tfidf_bert(
            row['query'], vectorizer, tfidf_matrix,
            model, bert_embeddings, df_index,
            tfidf_top_k=k)
        if answer.strip() == row['answer'].strip():
            correct += 1
    acc = correct / len(sample_df) * 100
    k_acc.append(acc)
    print(f'  TF-IDF k={k:2d} → Accuracy@1 = {acc:.1f}%')

plt.figure(figsize=(8, 4))
plt.plot(k_values, k_acc, 'ro-', linewidth=2, markersize=8)
plt.xlabel('TF-IDF Candidate Pool Size (k)')
plt.ylabel('Accuracy@1 (%)')
plt.title('TF-IDF + BERT — Effect of k on Accuracy',
          fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(
    os.path.join(RESULT_DIR, 'tfidf_bert_k_ablation.png'), dpi=150)
plt.show()

best_k = k_values[np.argmax(k_acc)]
print(f'\nBest k = {best_k}')

## Cell 10 — Evaluation Function

In [ ]:
def evaluate_tfidf_bert(df_test, vectorizer, tfidf_matrix,
                        model, bert_embeddings, df_index,
                        tfidf_top_k=10):
    correct_at1, correct_at3 = [], []
    rr_list, time_list       = [], []
    all_scores               = []

    for _, row in df_test.iterrows():
        query       = row['query']
        true_answer = row['answer']

        t_start = time.time()
        answer, score, _, top_k_res = retrieve_tfidf_bert(
            query, vectorizer, tfidf_matrix,
            model, bert_embeddings, df_index,
            tfidf_top_k=tfidf_top_k, final_top_k=5
        )
        elapsed = (time.time() - t_start) * 1000

        # Get top-5 answers
        top5_ans = []
        for q_text, _ in top_k_res:
            match = df_index[df_index['question'] == q_text]
            if len(match) > 0:
                top5_ans.append(match.iloc[0]['answer'])

        correct_at1.append(answer.strip() == true_answer.strip())
        correct_at3.append(
            true_answer.strip() in
            [a.strip() for a in top5_ans[:3]])

        rr = 0
        for rank, ans in enumerate(top5_ans, 1):
            if ans.strip() == true_answer.strip():
                rr = 1 / rank
                break
        rr_list.append(rr)
        time_list.append(elapsed)
        all_scores.append(score)

    return {
        'Accuracy@1'   : np.mean(correct_at1) * 100,
        'Accuracy@3'   : np.mean(correct_at3) * 100,
        'MRR'          : np.mean(rr_list),
        'Avg Score'    : np.mean(all_scores),
        'Avg Time (ms)': np.mean(time_list),
        'details'      : {
            'correct_at1': correct_at1,
            'scores'     : all_scores,
        }
    }

print('Evaluation function defined ✅')

## Cell 11 — Run Full Evaluation

In [ ]:
print(f'Running TF-IDF + BERT Hybrid (k={best_k})...')
print(f'  Index size : {len(df_index)} questions')
print(f'  Test size  : {len(df_test)} queries')

metrics = evaluate_tfidf_bert(
    df_test, vectorizer, tfidf_matrix,
    model, bert_embeddings, df_index,
    tfidf_top_k=best_k
)

print('\n' + '='*55)
print('  TF-IDF + BERT HYBRID RESULTS')
print('='*55)
print(f'  Accuracy@1    : {metrics["Accuracy@1"]:.2f}%')
print(f'  Accuracy@3    : {metrics["Accuracy@3"]:.2f}%')
print(f'  MRR           : {metrics["MRR"]:.4f}')
print(f'  Avg Score     : {metrics["Avg Score"]:.4f}')
print(f'  Avg Time (ms) : {metrics["Avg Time (ms)"]:.2f}')
print(f'  TF-IDF k      : {best_k}')
print('='*55)

## Cell 12 — Final Comparison: All Methods

In [ ]:
all_results = {
    'BM25'              : {'Accuracy@1': 87.23, 'Accuracy@3': 94.11,
                           'MRR': 0.9064, 'Avg Time (ms)': 0.49},
    'TF-IDF'            : {'Accuracy@1': 90.09, 'Accuracy@3': 98.04,
                           'MRR': 0.9399, 'Avg Time (ms)': 1.59},
    'Sentence-BERT'     : {'Accuracy@1': 92.59, 'Accuracy@3': 97.95,
                           'MRR': 0.9528, 'Avg Time (ms)': 45.88},
    'TF-IDF+BERT Hybrid': {
        'Accuracy@1'   : metrics['Accuracy@1'],
        'Accuracy@3'   : metrics['Accuracy@3'],
        'MRR'          : metrics['MRR'],
        'Avg Time (ms)': metrics['Avg Time (ms)'],
    }
}

compare_df = pd.DataFrame(all_results).T.round(4)
compare_df.index.name = 'Method'
compare_df.reset_index(inplace=True)
print('All Methods — Final Comparison:')
display(compare_df)

# Bar chart
methods = list(all_results.keys())
colors  = ['#e67e22', '#2980b9', '#8e44ad', '#27ae60']
x = np.arange(len(methods))
w = 0.28

acc1 = [all_results[m]['Accuracy@1'] for m in methods]
mrr  = [all_results[m]['MRR'] * 100  for m in methods]

fig, ax = plt.subplots(figsize=(12, 5))
b1 = ax.bar(x - w/2, acc1, w, label='Accuracy@1',
            color=colors, edgecolor='white', alpha=0.85)
b2 = ax.bar(x + w/2, mrr,  w, label='MRR × 100',
            color=colors, edgecolor='white', alpha=0.5,
            hatch='//')
ax.bar_label(b1, labels=[f'{v:.1f}%' for v in acc1], padding=3, fontsize=9)
ax.bar_label(b2, labels=[f'{v:.1f}' for v in mrr],   padding=3, fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(['BM25', 'TF-IDF', 'BERT', 'TF-IDF\n+BERT'], fontsize=10)
ax.set_ylabel('Score (%)')
ax.set_ylim(75, 110)
ax.set_title('UniMatch — All Methods Comparison\n'
             'Enhanced Dataset (1,120 queries)',
             fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(
    os.path.join(RESULT_DIR, 'all_methods_final.png'), dpi=150)
plt.show()
print('Plot saved ✅')

## Cell 13 — Save Results

In [ ]:
metrics_df = pd.DataFrame([{
    'Method'       : 'TF-IDF+BERT Hybrid',
    'Accuracy@1'   : round(metrics['Accuracy@1'], 2),
    'Accuracy@3'   : round(metrics['Accuracy@3'], 2),
    'MRR'          : round(metrics['MRR'], 4),
    'Avg Score'    : round(metrics['Avg Score'], 4),
    'Avg Time (ms)': round(metrics['Avg Time (ms)'], 2),
    'TF-IDF_k'     : best_k,
    'Index Size'   : len(df_index),
    'Test Queries' : len(df_test),
}])
metrics_df.to_csv(
    os.path.join(RESULT_DIR, 'tfidf_bert_hybrid_results.csv'),
    index=False)
compare_df.to_csv(
    os.path.join(RESULT_DIR, 'all_methods_final.csv'),
    index=False)

print('='*55)
print('  SAVED FILES')
print('='*55)
print(f'  results/tfidf_bert_hybrid_results.csv')
print(f'  results/all_methods_final.csv')
print(f'  results/all_methods_final.png')
print(f'  results/tfidf_bert_k_ablation.png')
print('='*55)
display(metrics_df)

best_m = compare_df.loc[compare_df['Accuracy@1'].idxmax(), 'Method']
print(f'\n🏆 Best method: {best_m}')

bert_acc  = 92.59
hybrid_acc = metrics['Accuracy@1']
delta = hybrid_acc - bert_acc
if delta > 0:
    print(f'✅ TF-IDF+BERT beats BERT alone by {delta:.2f}%!')
else:
    print(f'ℹ️ BERT alone still better by {abs(delta):.2f}%')

print(f'\nNext → 05_evaluation_comparison.ipynb')

In [ ]:
import pickle
import numpy as np

# حفظ TF-IDF vectorizer
with open('/content/drive/MyDrive/Student_chatbot/models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

# حفظ TF-IDF matrix
with open('/content/drive/MyDrive/Student_chatbot/models/tfidf_matrix.pkl', 'wb') as f:
    pickle.dump(tfidf_matrix, f)

# حفظ BERT embeddings (موجود مسبقاً)
# bert_embeddings_v2.npy — already saved

# حفظ dataset
df_index.to_csv(
    '/content/drive/MyDrive/Student_chatbot/models/faq_index.csv',
    index=False, encoding='utf-8-sig')

print('All models saved ✅')